# Tabla de resultados de películas (incremental)

Versión incremental de `05. results_movie`: toma las películas del lote `p_file_date` y actualiza `movie_gold.results_movie` con un `MERGE` en SQL.

### Configuración

Lee `p_file_date` (fecha del lote).

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

### Contexto

Revisa catálogo y esquemas y selecciona `movie_silver` como esquema activo.

In [0]:
%sql
SELECT current_catalog()

In [0]:
%sql
SHOW SCHEMAS

In [0]:
%sql
SELECT current_schema()

In [0]:
%sql
USE movie_silver

In [0]:
%sql
SELECT current_schema()

### Vista fuente del lote

Crea la vista temporal `view_results_movie` con las películas del lote y sus países y compañías.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW view_results_movie
    AS
    SELECT
    M.movie_id, C.country_id, PCO.company_id,
    M.title,
    M.year_release_date,
    C.country_name,
    PCO.company_name,
    M.budget,
    M.revenue
    FROM movie_silver.movies as M
    INNER JOIN movie_silver.production_countries as PC ON M.movie_id = PC.movie_id
    INNER JOIN movie_silver.countries AS C ON PC.country_id = C.country_id
    INNER JOIN movie_silver.movies_companies AS MC ON M.movie_id = MC.movie_id
    INNER JOIN movie_silver.production_companies AS PCO ON MC.company_id = PCO.company_id
    WHERE M.file_date = '{v_file_date}'
    """
)

## Utilizando merge

### Tabla destino

Crea `movie_gold.results_movie` si no existe, con `created_date` y `updated_date` para auditar las cargas.

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS movie_gold.results_movie
          (
            year_release_date INT,
            country_name STRING,
            company_name STRING,
            budget FLOAT,
            revenue FLOAT,
            movie_id STRING,
            country_id INT,
            company_id INT,
            created_date DATE,
            updated_date DATE
          )
          USING DELTA
          """)

### MERGE

Actualiza las filas que coinciden por `movie_id`, `country_id` y `company_id` (fija `updated_date`) e inserta las nuevas con `created_date`.

In [0]:
%sql
MERGE INTO movie_gold.results_movie AS trgt
USING view_results_movie AS src
ON trgt.movie_id = src.movie_id AND trgt.country_id = src.country_id AND trgt.company_id = src.company_id
WHEN MATCHED THEN
  UPDATE SET
    trgt.year_release_date = src.year_release_date,
    trgt.country_name = src.country_name,
    trgt.company_name = src.company_name,
    trgt.budget = src.budget,
    trgt.revenue = src.revenue,
    trgt.updated_date = current_timestamp
WHEN NOT MATCHED
  THEN INSERT (movie_id, country_id, company_id, year_release_date, country_name, company_name,budget, revenue, created_date)
  VALUES (movie_id, country_id, company_id, year_release_date, country_name, company_name,budget, revenue, current_timestamp)

In [0]:
%sql
SELECT created_date, COUNT(*)
FROM movie_gold.results_movie
GROUP BY created_date
ORDER BY created_date